# Lab 0: Getting Set Up — Jupyter, Python & GeoPandas
## ENCN205 — Applied Data Analysis for Civil and Natural Systems
### University of Canterbury | Te Whare Wānanga o Waitaha

**Duration:** 30–45 minutes
**When:** Week 8 (online, self-paced) — complete **before Lab 1**
**Format:** Ungraded setup module — sign-off submitted to LEARN

---

### Learning Objectives
By the end of this module you will be able to:
1. Launch JupyterLab and run notebook cells
2. Verify that your Python geospatial environment is correctly installed
3. Create, reproject, and map your first spatial dataset
4. Know where to get help if something isn't working

### Before you open this notebook
Follow **Lab0_Instructions** (on LEARN) to install Miniforge/Anaconda and create the
`encn205` environment. If you can read this text inside JupyterLab, you've already
done the hard part — well done!

> **Why this matters:** Labs 1–4 are assessed and assume a working environment.
> Lab 0 exists so that in Lab 1 you spend your time doing GIS, not fixing installations.

## 1. Welcome to Jupyter

A Jupyter notebook is made of **cells**. There are two kinds:

| Cell type | Contains | Example |
|-----------|----------|---------|
| **Markdown** | Formatted text, tables, images | This cell! |
| **Code** | Python that you can run | The cell below |

To run a cell: click on it, then press **Shift + Enter** (or the ▶ button in the toolbar).
The notebook runs cells in the order *you* run them, and remembers variables between cells.

Try it — run the cell below:

In [ ]:
# This is a code cell. Run it with Shift + Enter.
print("Kia ora! Welcome to ENCN205 GIS.")

# Variables persist between cells
course = "ENCN205"
weeks_of_gis = 6
print(f"{course} includes {weeks_of_gis} weeks of geospatial data analysis.")

### Your turn

Edit the cell below: replace the name with your own, then run it.
(Editing and re-running cells is how you'll complete all the lab exercises.)

In [ ]:
# EDIT ME: put your name between the quotes, then press Shift + Enter
my_name = "Your Name Here"

print(f"Ka pai, {my_name} — you can run and edit notebook cells.")

## 2. Check Your Environment

The GIS labs use the Python geospatial stack. The cell below checks that every
package the labs need is installed in your `encn205` environment.

**Every line should show a green tick (✅).** If you see any ❌, jump to the
Troubleshooting section of Lab0_Instructions — the fix is usually one
`conda install` command.

In [ ]:
# Check that all packages used in Labs 1-4 are installed
import sys
import importlib

print(f"Python version: {sys.version.split()[0]}", end="")
py_ok = sys.version_info >= (3, 10)
print("  ✅" if py_ok else "  ❌ (need Python 3.10 or newer)")
print("-" * 55)

PACKAGES = [
    ("geopandas",   "Spatial DataFrames (the core of every lab)"),
    ("pandas",      "Tabular data"),
    ("numpy",       "Arrays and raster maths"),
    ("matplotlib",  "Plotting and maps"),
    ("shapely",     "Geometry objects"),
    ("pyproj",      "Coordinate reference systems"),
    ("pyogrio",     "Fast spatial file reading"),
    ("fiona",       "Spatial file access"),
    ("folium",      "Interactive web maps"),
    ("geodatasets", "Sample datasets"),
]

all_ok = py_ok
for name, purpose in PACKAGES:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", "installed")
        print(f"✅ {name:<12} {version:<10} — {purpose}")
    except ImportError:
        all_ok = False
        print(f"❌ {name:<12} MISSING    — run: conda install -c conda-forge {name}")

print("-" * 55)
if all_ok:
    print("✅ Environment check PASSED — carry on to Section 3.")
else:
    print("❌ Something is missing — see Troubleshooting in Lab0_Instructions.")

## 3. Your First Map

Let's do a tiny version of what you'll do all through the GIS block:
**create spatial data, give it a coordinate reference system (CRS), reproject it, and map it.**

Don't worry about understanding every line yet — Lectures 1–4 and Lab 1 will cover all
of this properly. Today the goal is simply to see it run on your machine.

We'll use four Christchurch landmarks, with coordinates in longitude/latitude
(WGS 84, the system GPS uses).

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Four Christchurch landmarks: (longitude, latitude) in WGS 84
landmarks = pd.DataFrame({
    "name": ["UC Campus", "Cathedral Square", "New Brighton Pier", "Lyttelton Port"],
    "lon":  [172.5833, 172.6362, 173.0060, 172.7220],
    "lat":  [-43.5226, -43.5309, -43.5065, -43.6060],
})

# Turn the table into a GeoDataFrame — a table where every row has a geometry
gdf = gpd.GeoDataFrame(
    landmarks,
    geometry=gpd.points_from_xy(landmarks.lon, landmarks.lat),
    crs="EPSG:4326",   # WGS 84 (longitude/latitude)
)

print(gdf)
print(f"\nCRS: {gdf.crs.name}")

### Reproject to NZTM2000

Longitude/latitude is measured in **degrees**, which is awkward for engineering —
you can't buffer a pipe by 0.0001 degrees. In this course all analysis uses
**NZTM2000 (EPSG:2193)**, New Zealand's standard projected CRS, where coordinates
are in **metres**.

One line converts the whole dataset:

In [ ]:
# Reproject from WGS 84 (degrees) to NZTM2000 (metres)
gdf_nztm = gdf.to_crs("EPSG:2193")

print(gdf_nztm[["name", "geometry"]])
print(f"\nCRS: {gdf_nztm.crs.name}")

# Now that coordinates are in metres, distances make sense:
campus = gdf_nztm[gdf_nztm.name == "UC Campus"].geometry.iloc[0]
gdf_nztm["km_from_UC"] = (gdf_nztm.distance(campus) / 1000).round(1)
print("\nDistance from UC Campus:")
print(gdf_nztm[["name", "km_from_UC"]].to_string(index=False))

In [ ]:
# Map the landmarks, with a 2 km buffer around campus
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 7))

# 2 km buffer around campus — possible because our units are metres
gpd.GeoSeries([campus.buffer(2000)], crs="EPSG:2193").plot(
    ax=ax, color="lightblue", edgecolor="steelblue", alpha=0.6)

gdf_nztm.plot(ax=ax, color="crimson", markersize=80, zorder=3)

for _, row in gdf_nztm.iterrows():
    ax.annotate(row["name"], (row.geometry.x, row.geometry.y),
                xytext=(8, 5), textcoords="offset points", fontsize=10)

ax.set_title("Your first GIS output: Christchurch landmarks (NZTM2000)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain")
plt.tight_layout()
plt.show()

print("✅ If you can see a map with four red points and a blue circle, your")
print("   geospatial stack works end-to-end.")

## 4. Bonus: An Interactive Map *(optional — needs internet)*

Static maps go in reports; interactive maps are great for exploring data.
GeoPandas can make one in a single line with `.explore()`, using the folium library.

The background map tiles load from the internet, so this cell needs a connection —
if it doesn't display, that's fine, it's not required for Lab 1.

In [ ]:
# One line: an interactive map you can pan and zoom (uses folium under the hood)
gdf.explore(marker_kwds={"radius": 8}, tooltip="name", zoom_start=11)

## 5. Final Readiness Check

Run the cell below. If everything above worked, you'll get the all-clear for Lab 1.

In [ ]:
# Final check: imports, versions, and a round-trip CRS transformation
import sys
import importlib

checks = []

# 1. Python version
checks.append(("Python ≥ 3.10", sys.version_info >= (3, 10)))

# 2. All required packages import
required = ["geopandas", "pandas", "numpy", "matplotlib", "shapely",
            "pyproj", "pyogrio", "fiona", "folium", "geodatasets"]
missing = []
for name in required:
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(name)
checks.append((f"All {len(required)} packages installed", not missing))

# 3. GeoPandas is a recent version
import geopandas as gpd
major = int(gpd.__version__.split(".")[0])
checks.append((f"GeoPandas version ({gpd.__version__})", major >= 1))

# 4. CRS machinery works: WGS84 -> NZTM2000 lands where it should
from shapely.geometry import Point
uc = gpd.GeoSeries([Point(172.5833, -43.5226)], crs="EPSG:4326").to_crs("EPSG:2193")
x, y = uc.iloc[0].x, uc.iloc[0].y
crs_ok = abs(x - 1_566_325) < 1_000 and abs(y - 5_181_064) < 1_000
checks.append(("CRS reprojection works (WGS84 → NZTM2000)", crs_ok))

print("=" * 55)
for label, ok in checks:
    print(f"{'✅' if ok else '❌'}  {label}")
print("=" * 55)

if all(ok for _, ok in checks):
    print()
    print("   🎉  YOU ARE READY FOR LAB 1  🎉")
    print()
    print("   One more step: complete Lab0_SignOff.html and upload your PNG to LEARN.")
    print("   See you in Lab 1: Reading, Exploring, and Projecting Spatial Data.")
else:
    print()
    print("   Not quite there yet — see the Troubleshooting section of")
    print("   Lab0_Instructions, or bring your laptop to the first lab session.")

## 6. Lab Sign-Off

### Assessment: Lab Sign-Off

**This lab is not complete until you have submitted your sign-off to LEARN.**

**Step 1 — Self-check.** Make sure you can do each of these:

- [ ] Ran a code cell with Shift + Enter and edited a cell of your own
- [ ] Every package in the environment check shows a green tick
- [ ] The Christchurch landmarks map displayed four red points and a blue buffer
- [ ] The final readiness check printed 'YOU ARE READY FOR LAB 1'

**Step 2 — Take the sign-off quiz.**

1. Download **`Lab0_SignOff.html`** from LEARN and open it in your web browser.
2. Enter your full name and University ID, then answer the 5 questions.
3. You need **4/5** to pass. You can retake it as often as you like — each
   attempt draws a new set of questions, and every answer comes with an explanation.
4. When you pass, click **Download Sign-Off PNG**. This saves a file called
   `ENCN205_Lab0_<yourID>.png` to your computer.

**Step 3 — Submit.** Upload `ENCN205_Lab0_<yourID>.png` to the **Lab 0 Sign-Off**
submission on LEARN.

> Your PNG records your name, ID, score, and a timestamp, encoded in the QR code.
> It is your proof of completion — show it to your tutor if they ask.


## 7. If Something Went Wrong

Don't panic — installation issues are normal and quick to fix. In order of preference:

1. **Check the Troubleshooting table** in Lab0_Instructions (on LEARN) — it covers the
   common errors (`conda` not recognised, `ImportError`, wrong kernel selected, etc.).
2. **Post on the course LEARN forum** with the exact error message (screenshots help).
3. **Bring your laptop to your first lab session** — tutors will help you get set up
   before the assessed work starts.

---

### What's next?

| When | What |
|------|------|
| Week 9 lectures | Coordinate reference systems & spatial relationships |
| **Lab 1** | Reading, exploring, and projecting spatial data *(assessed, 1%)* |

*ENCN205 — University of Canterbury | GIS component. Lab 0 is an ungraded setup module, but the sign-off must be submitted to LEARN.*